## Carga de imágenes

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el espécimen; el resto queda negro.

    Parámetros
    ----------
    hdr_path : str
        Ruta al archivo .hdr de la imagen hiperespectral.

    r_nm, g_nm, b_nm : float
        Longitudes de onda usadas para construir la pseudo-RGB.

    grow_pixels : int
        Número de píxeles para agrandar ligeramente el contorno del espécimen.
        Útil para evitar cortar tejido.

    show_original : bool
        Si True, plotea la pseudo-RGB original.

    show_result : bool
        Si True, plotea la imagen con solo el espécimen.

    return_mask : bool
        Si True, devuelve también la máscara binaria del espécimen.

    Returns
    -------
    specimen_only_rgb : np.ndarray
        Imagen RGB uint8 con solo el espécimen visible y el resto negro.

    Opcionalmente:
    mask_specimen : np.ndarray
        Máscara uint8 del espécimen, con valores 0 y 255.
    """

    # ============================================================
    # 1. Cargar cubo HSI
    # ============================================================
    img = open_image(hdr_path)
    cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array(
        [float(w) for w in img.metadata["wavelength"]],
        dtype=np.float32
    )

    # ============================================================
    # 2. Funciones internas
    # ============================================================
    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        idx = np.argmin(np.abs(wavelengths_arr - target_nm))
        return idx

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)

        lo = np.percentile(ch, p_low)
        hi = np.percentile(ch, p_high)

        if hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)

        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)

        return ch

    # ============================================================
    # 3. Crear pseudo-RGB
    # ============================================================
    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    r = robust_normalize(cube[:, :, r_idx])
    g = robust_normalize(cube[:, :, g_idx])
    b = robust_normalize(cube[:, :, b_idx])

    pseudo_rgb = np.stack([r, g, b], axis=-1)
    rgb_u8 = (np.clip(pseudo_rgb, 0, 1) * 255).astype(np.uint8)

    # ============================================================
    # 4. Detectar caja azul
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    R = rgb_u8[:, :, 0].astype(np.int16)
    G = rgb_u8[:, :, 1].astype(np.int16)
    B = rgb_u8[:, :, 2].astype(np.int16)

    V = hsv[:, :, 2]

    lower_blue = np.array([85, 10, 40], dtype=np.uint8)
    upper_blue = np.array([125, 180, 255], dtype=np.uint8)

    mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)

    mask_dominance = (
        ((B - R) > 8) &
        ((G - R) > 3) &
        (V > 50)
    )

    mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

    kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))

    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
    mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
    mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

    if np.count_nonzero(mask_blue) == 0:
        raise ValueError("No se detectó la caja azul. Revisa los umbrales HSV.")

    # ============================================================
    # 5. Quedarse con el componente azul más grande
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_blue,
        connectivity=8
    )

    if num_labels <= 1:
        raise ValueError("No se encontró ningún componente azul válido.")

    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_label = 1 + np.argmax(areas)

    mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

    # ============================================================
    # 6. Contorno interno de la caja azul = espécimen
    # ============================================================
    contours, hierarchy = cv2.findContours(
        mask_box_blue,
        cv2.RETR_CCOMP,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if hierarchy is None:
        raise ValueError("No se encontraron contornos en la máscara de la caja azul.")

    hierarchy = hierarchy[0]

    inner_contours = [
        cnt for cnt, h in zip(contours, hierarchy)
        if h[3] != -1
    ]

    if len(inner_contours) == 0:
        raise ValueError("No se encontró ningún hueco interno en la caja azul.")

    specimen_contour = max(inner_contours, key=cv2.contourArea)

    mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)
    cv2.drawContours(
        mask_specimen,
        [specimen_contour],
        -1,
        255,
        thickness=cv2.FILLED
    )

    # Suavizar pequeñas irregularidades
    kernel_smooth = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask_specimen = cv2.morphologyEx(
        mask_specimen,
        cv2.MORPH_CLOSE,
        kernel_smooth
    )

    # Agrandar ligeramente el contorno
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )

        mask_specimen = cv2.dilate(
            mask_specimen,
            kernel_grow,
            iterations=1
        )

    # ============================================================
    # 7. Aplicar máscara: espécimen visible, resto negro
    # ============================================================
    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if show_original and show_result:
        plt.figure(figsize=(12, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title("Solo espécimen")
        plt.axis("off")
        plt.show()

    if return_mask:
        return specimen_only_rgb, mask_specimen

    return specimen_only_rgb

In [ ]:
hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"

specimen_only_rgb, mask_specimen = extract_specimen_only_from_hsi_path(
    hdr_path,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=True
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(specimen_only_rgb)
plt.title("Specimen only RGB")
plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import openslide


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """
    Extrae el tejido de una H&E tipo .mrxs usando el contorno externo del tejido.

    Por defecto devuelve solo:
        tissue_only_rgb

    Opcionalmente puede devolver:
        mask_tissue
        info

    Parameters
    ----------
    slide_path : str or Path
        Ruta al archivo .mrxs.

    max_dim : int
        Tamaño máximo del lado largo usado para leer la imagen a baja resolución.

    sat_thresh : int
        Umbral de saturación HSV. Más bajo detecta tejido más pálido.

    val_thresh : int
        Umbral de brillo HSV. Más alto permite zonas más claras.

    od_thresh : float
        Umbral de optical density. Más bajo detecta tejido muy claro.

    min_area : int
        Área mínima de componentes a conservar.

    open_kernel_size : int
        Kernel para eliminar ruido pequeño.

    close_kernel_size : int
        Kernel para cerrar huecos y unir zonas del tejido.

    grow_pixels : int
        Dilatación final del contorno para no cortar tejido.

    use_convex_hull : bool
        Si True, usa convex hull. Es más conservador, puede meter más fondo.

    show_original : bool
        Plotea la H&E original leída.

    show_result : bool
        Plotea el resultado final.

    debug : bool
        Muestra pasos intermedios.

    return_mask : bool
        Si True, devuelve también la máscara final.

    return_info : bool
        Si True, devuelve también información auxiliar.
    """

    slide_path = Path(slide_path)

    if not slide_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {slide_path}")

    # ============================================================
    # 1. Cargar slide a baja resolución
    # ============================================================
    slide = openslide.OpenSlide(str(slide_path))
    level_dims = slide.level_dimensions

    chosen_level = None
    for i, (w, h) in enumerate(level_dims):
        if max(w, h) <= max_dim:
            chosen_level = i
            break

    if chosen_level is None:
        chosen_level = len(level_dims) - 1

    level_w, level_h = level_dims[chosen_level]

    rgb_pil = slide.read_region(
        location=(0, 0),
        level=chosen_level,
        size=(level_w, level_h)
    ).convert("RGB")

    slide.close()

    rgb_u8 = np.array(rgb_pil, dtype=np.uint8)

    # ============================================================
    # 2. Mapas HSV y optical density
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32)
    od = -np.log((rgb_float + 1.0) / 255.0)
    od_sum = od.sum(axis=2)

    # ============================================================
    # 3. Máscara inicial: tejido teñido o tejido pálido
    # ============================================================
    mask_sat = (
        (S > sat_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_od = (
        (od_sum > od_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_raw = cv2.bitwise_or(mask_sat, mask_od)

    # ============================================================
    # 4. Morfología
    # ============================================================
    mask_morph = mask_raw.copy()

    if open_kernel_size > 0:
        kernel_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (open_kernel_size, open_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size > 0:
        kernel_close = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_kernel_size, close_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_CLOSE, kernel_close)

    # ============================================================
    # 5. Componentes conectados
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_morph,
        connectivity=8
    )

    mask_components = np.zeros_like(mask_morph, dtype=np.uint8)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask_components[labels == label_id] = 255

    if np.count_nonzero(mask_components) == 0:
        raise ValueError(
            "No se detectó tejido. Prueba sat_thresh=5, od_thresh=0.012, val_thresh=255."
        )

    # ============================================================
    # 6. Extraer contorno externo y rellenarlo
    # ============================================================
    contours, _ = cv2.findContours(
        mask_components,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        raise ValueError("No se encontraron contornos de tejido.")

    # Normalmente el tejido principal es el mayor contorno
    largest_contour = max(contours, key=cv2.contourArea)

    mask_contour = np.zeros_like(mask_components, dtype=np.uint8)

    if use_convex_hull:
        hull = cv2.convexHull(largest_contour)
        cv2.drawContours(mask_contour, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        cv2.drawContours(mask_contour, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Dilatar ligeramente para no cortar borde
    mask_final = mask_contour.copy()

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    # ============================================================
    # 7. Aplicar máscara: tejido visible, resto negro
    # ============================================================
    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if debug:
        od_vis = (od_sum - od_sum.min()) / (od_sum.max() - od_sum.min() + 1e-8)

        plt.figure(figsize=(18, 10))

        plt.subplot(2, 4, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(2, 4, 2)
        plt.imshow(S, cmap="gray")
        plt.title("Saturación HSV")
        plt.axis("off")

        plt.subplot(2, 4, 3)
        plt.imshow(od_vis, cmap="gray")
        plt.title("Optical density")
        plt.axis("off")

        plt.subplot(2, 4, 4)
        plt.imshow(mask_raw, cmap="gray")
        plt.title("Máscara inicial")
        plt.axis("off")

        plt.subplot(2, 4, 5)
        plt.imshow(mask_morph, cmap="gray")
        plt.title("Después morfología")
        plt.axis("off")

        plt.subplot(2, 4, 6)
        plt.imshow(mask_components, cmap="gray")
        plt.title("Componentes filtrados")
        plt.axis("off")

        plt.subplot(2, 4, 7)
        plt.imshow(mask_final, cmap="gray")
        plt.title("Máscara final por contorno")
        plt.axis("off")

        plt.subplot(2, 4, 8)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original and show_result:
        plt.figure(figsize=(10, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")
        plt.show()

    # ============================================================
    # 9. Return limpio
    # ============================================================
    outputs = [tissue_only_rgb]

    if return_mask:
        outputs.append(mask_final)

    if return_info:
        info = {
            "chosen_level": chosen_level,
            "read_dimensions": (level_w, level_h),
            "sat_thresh": sat_thresh,
            "val_thresh": val_thresh,
            "od_thresh": od_thresh,
            "grow_pixels": grow_pixels,
            "use_convex_hull": use_convex_hull,
            "mask_area_px": int(np.count_nonzero(mask_final)),
        }
        outputs.append(info)

    if len(outputs) == 1:
        return tissue_only_rgb

    return tuple(outputs)

In [ ]:
slide_path = Path(r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs")

tissue_only_rgb = extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    show_original=True,
    show_result=True,
    debug=False
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(tissue_only_rgb)
plt.title("Specimen only RGB")
plt.axis("off")
plt.show()

## Imágenes q tenemos

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(specimen_only_rgb)
plt.title("Specimen only RGB")
plt.axis("off")
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(tissue_only_rgb)
plt.title("Tissue only RGB")
plt.axis("off")
plt.show()

## Tamaño de las imágenes

In [ ]:
import openslide
from pathlib import Path

slide_path = Path(r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs")

slide = openslide.OpenSlide(str(slide_path))

print("Dimensiones level 0:", slide.dimensions)
print("Downsamples:", slide.level_downsamples)

print("\nPropiedades relevantes:")
for k, v in slide.properties.items():
    kl = k.lower()
    if "mpp" in kl or "micron" in kl or "pixel" in kl or "objective" in kl:
        print(k, ":", v)

slide.close()

In [ ]:
from spectral import open_image

hdr_path = r"Datos\SB013\SB013\SB013_001\SB013_nrm.hdr"

img = open_image(hdr_path)

print("HSI shape:", img.shape)

print("\nMetadata relevante:")
for k, v in img.metadata.items():
    kl = k.lower()
    if (
        "pixel" in kl or
        "spatial" in kl or
        "resolution" in kl or
        "fov" in kl or
        "mm" in kl or
        "micron" in kl or
        "scale" in kl
    ):
        print(k, ":", v)

## Técnicas deep

In [ ]:
import cv2
import torch
import kornia as K
import kornia.feature as KF
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def resize_max_side(img, max_side=1024):
    h, w = img.shape[:2]
    scale = max_side / max(h, w)
    if scale >= 1:
        return img, 1.0
    new_w = int(w * scale)
    new_h = int(h * scale)
    out = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return out, scale


def rgb_to_gray_tensor(img_rgb):
    # img_rgb: uint8, shape (H, W, 3)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)   # (H, W)
    gray = gray.astype(np.float32) / 255.0             # [0,1]
    gray = torch.from_numpy(gray)[None, None, :, :]    # (1,1,H,W)
    return gray

In [ ]:
plt.figure(figsize=(12,6))
plt.subplot(1,2,1)
plt.imshow(specimen_only_rgb)
plt.title("HSI pseudo-RGB")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(tissue_only_rgb)
plt.title("WSI")
plt.axis("off")
plt.show()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
matcher = KF.LoFTR(pretrained="outdoor").to(device).eval()

img0 = (specimen_only_rgb).to(device)
img1 = (tissue_only_rgb).to(device)

with torch.no_grad():
    input_dict = {"image0": img0, "image1": img1}
    correspondences = matcher(input_dict)

mkpts0 = correspondences["keypoints0"].cpu().numpy()
mkpts1 = correspondences["keypoints1"].cpu().numpy()
mconf  = correspondences["confidence"].cpu().numpy()

print("Número de matches:", len(mkpts0))

In [ ]:
def draw_matches(img0, img1, pts0, pts1, max_draw=80):
    n = min(len(pts0), max_draw)
    idx = np.linspace(0, len(pts0)-1, n, dtype=int) if len(pts0) > 0 else []

    p0 = pts0[idx] if len(pts0) > 0 else np.empty((0,2))
    p1 = pts1[idx] if len(pts1) > 0 else np.empty((0,2))

    h0, w0 = img0.shape[:2]
    h1, w1 = img1.shape[:2]
    H = max(h0, h1)
    canvas = np.zeros((H, w0 + w1, 3), dtype=np.uint8)
    canvas[:h0, :w0] = img0
    canvas[:h1, w0:w0+w1] = img1

    for (x0, y0), (x1, y1) in zip(p0, p1):
        c = tuple(np.random.randint(0, 255, 3).tolist())
        pt0 = (int(x0), int(y0))
        pt1 = (int(x1) + w0, int(y1))
        cv2.circle(canvas, pt0, 3, c, -1)
        cv2.circle(canvas, pt1, 3, c, -1)
        cv2.line(canvas, pt0, pt1, c, 1)

    return canvas

vis = draw_matches(specimen_only_rgb, tissue_only_rgb, mkpts0, mkpts1, max_draw=60)

plt.figure(figsize=(16,8))
plt.imshow(vis)
plt.title("Matches LoFTR")
plt.axis("off")
plt.show()